In [1]:
from src import *
import numpy as np
from tqdm import tqdm
from time import sleep
import os

from myutils import email_notify


from datetime import datetime
today = datetime.strftime(datetime.today(), '%Y%m%d')
del datetime

In [2]:
today

'20241106'

In [3]:
# with RaspiLED() as led:
#     led.check()

In [4]:
np.linspace(0.1, 0.4, 61)

array([0.1  , 0.105, 0.11 , 0.115, 0.12 , 0.125, 0.13 , 0.135, 0.14 ,
       0.145, 0.15 , 0.155, 0.16 , 0.165, 0.17 , 0.175, 0.18 , 0.185,
       0.19 , 0.195, 0.2  , 0.205, 0.21 , 0.215, 0.22 , 0.225, 0.23 ,
       0.235, 0.24 , 0.245, 0.25 , 0.255, 0.26 , 0.265, 0.27 , 0.275,
       0.28 , 0.285, 0.29 , 0.295, 0.3  , 0.305, 0.31 , 0.315, 0.32 ,
       0.325, 0.33 , 0.335, 0.34 , 0.345, 0.35 , 0.355, 0.36 , 0.365,
       0.37 , 0.375, 0.38 , 0.385, 0.39 , 0.395, 0.4  ])

In [5]:
sampling_rate = 20 #Hz
freq_list = np.linspace(0.1, 0.4, 61)

pic_time = (1 / (sampling_rate * freq_list) / 2 * 1e6).astype(int) #us

A = 5
samples = 50
repeat = 200

interval = 50e-3 #s, 50 ms
exposure_time = 500e-6 #s, 500 us
timeout_milisec = 3000 #ms, 3000 ms

measurement = 'DI'

In [6]:
############
# NO NOISE #
############

@email_notify('hcnzj@qq.com')
def main():
    with EasyDcam() as dcam, EasyALP4() as alp:
        for i, picture_time in enumerate(pic_time):
            print(f'({i + 1}): Current sensor temperature is {dcam.ez_temperature()}')
            if dcam.ez_temperature() >= -30:
                raise RuntimeError("qCMOS's temperature is too high.")

            ground_truth = np.round(freq_list[i], 5)

            alp.ez_load_seq([alp.ez_single_pixel(0), alp.ez_single_pixel(A)], picture_time)
            dcam.ez_exposure_time(exposure_time)
            dcam.ez_triggersource_masterpluse(samples, interval)

            if measurement.upper() == 'SPADE':
                dcam.ez_roi(**SPADE.ROI)
            elif measurement.upper() == 'DI':
                dcam.ez_roi(**DI.ROI)

            raw, timestamp = [], []
            for _ in tqdm(range(repeat)):
                dcam.buf_alloc(samples)
                dcam.cap_snapshot()

                alp.Run()
                sleep(1e-6)
                dcam.cap_firetrigger()

                dcam.ez_wait_capture(timeout_milisec)

                dcam.cap_stop()
                alp.Halt()

                raw_, timestamp_ = [], []
                for frame in range(samples):
                    framedata_ = dcam.ez_read_buf(frame)
                    raw_.append(framedata_[0])
                    timestamp_.append(framedata_[1])

                dcam.buf_release()

                raw.append(raw_)
                timestamp.append(timestamp_)

            raw = np.array(raw)
            timestamp = np.array(timestamp)

            if not os.path.exists(f'__raw__/{today}'):
                os.makedirs(f'__raw__/{today}')
            if not os.path.exists(f'__estimates__/{today}'):
                os.makedirs(f'__estimates__/{today}')

            np.save(f'./__raw__/{today}/{measurement.lower()}_{ground_truth}_raw.npy', raw)
            np.save(f'./__raw__/{today}/{measurement.lower()}_{ground_truth}_timestamp.npy', timestamp)

            metadata = MetaData(measurement, ground_truth, A*DMD.PIXEL_SIZE/2, timestamp)
            est = FrequencyEstimation.FromRaw(np.array(raw), metadata)
            est.savez(f'./__estimates__/{today}/{measurement.lower()}_{ground_truth}.npz')

            print(f'({i + 1}): {dcam.lasterr()}')


if __name__ == '__main__':
    main()

qCMOS found, current sensor temperature is -37.0.
DMD found, resolution = 1024 x 768.
(1): Current sensor temperature is -37.0


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


(1): 1
(2): Current sensor temperature is -36.0


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


(2): 1
(3): Current sensor temperature is -36.0


...

(59): 1
(60): Current sensor temperature is -36.0


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


(60): 1
(61): Current sensor temperature is -36.0


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


(61): 1
EasyALP4 exited
EasyDcam exited


In [7]:
raise RuntimeError('STOP HERE')

RuntimeError: STOP HERE